# 03 Train NAND Best
Best-Path Training (InfoNCE, 818->1024->256).

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

RUN_ID = "replace_with_run_id_from_00"
DEVICE = "auto"


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
import numpy as np
import pandas as pd

from src.common.config import load_notebook_context, build_run_dirs
from src.common.io_schema import read_parquet
from src.approaches.nand.train import train_nand_across_seeds

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]

RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for key in ["subset_cache", "embeddings", "checkpoints", "metrics"]:
    RUN_DIRS[key].mkdir(parents=True, exist_ok=True)

training_cfg = MODEL["training"]
subset_dir = RUN_DIRS["subset_cache"]
emb_dir = RUN_DIRS["embeddings"]
ckpt_dir = RUN_DIRS["checkpoints"]
metrics_dir = RUN_DIRS["metrics"]

lspo_mentions = read_parquet(subset_dir / f"lspo_mentions_{RUN_STAGE}.parquet")
lspo_pairs = read_parquet(subset_dir / f"lspo_pairs_{RUN_STAGE}.parquet")
lspo_chars = np.load(emb_dir / f"lspo_chars2vec_{RUN_STAGE}.npy")
lspo_text = np.load(emb_dir / f"lspo_specter_{RUN_STAGE}.npy")

print("Model:", MODEL.get("name"))
print("Loss:", training_cfg.get("loss"))
print("Architecture:", f"{training_cfg['input_dim']} -> {training_cfg['hidden_dim']} -> {training_cfg['output_dim']}")


Model: nand_best
Loss: infonce
Architecture: 818 -> 1024 -> 256


In [3]:
default_seeds = training_cfg.get("seeds", [1,2,3,4,5])
seeds = [default_seeds[0]] if RUN_STAGE in {"smoke", "mini"} else default_seeds

manifest_path = RUN_DIRS["metrics"] / "03_train_manifest.json"
manifest = train_nand_across_seeds(
    mentions=lspo_mentions,
    pairs=lspo_pairs,
    chars2vec=lspo_chars,
    text_emb=lspo_text,
    model_config=training_cfg,
    seeds=seeds,
    run_id=RUN_ID,
    output_dir=ckpt_dir,
    metrics_output=manifest_path,
    device=DEVICE,
)
manifest


{'run_id': 'replace_with_run_id_from_00',
 'runs': [{'seed': 1,
   'checkpoint': 'C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\checkpoints\\replace_with_run_id_from_00\\replace_with_run_id_from_00_seed1.pt',
   'threshold': -1.0,
   'val_stats': {'f1': 0.9850746268656716,
    'precision': 0.9705882352941176,
    'recall': 1.0,
    'accuracy': 0.9705882352941176},
   'test_metrics': {'f1': 0.9882352941176471,
    'precision': 0.9767441860465116,
    'recall': 1.0,
    'accuracy': 0.9767441860465116}}],
 'best_seed': 1,
 'best_checkpoint': 'C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\checkpoints\\replace_with_run_id_from_00\\replace_with_run_id_from_00_seed1.pt',
 'best_threshold': -1.0,
 'best_val_f1': 0.9850746268656716}

In [4]:
rows = []
for r in manifest["runs"]:
    rows.append({
        "seed": r["seed"],
        "checkpoint": r["checkpoint"],
        "threshold": r["threshold"],
        "val_f1": r["val_stats"].get("f1"),
        "val_precision": r["val_stats"].get("precision"),
        "val_recall": r["val_stats"].get("recall"),
        "test_f1": r["test_metrics"].get("f1"),
        "test_precision": r["test_metrics"].get("precision"),
        "test_recall": r["test_metrics"].get("recall"),
    })

pd.DataFrame(rows).sort_values("val_f1", ascending=False)


,seed,checkpoint,threshold,val_f1,val_precision,val_recall,test_f1,test_precision,test_recall
0,1,C:\Users\rapha\Documents\Studium\Promotionsstu...,-1.0,0.985075,0.970588,1.0,0.988235,0.976744,1.0
